# Simulating Extended Source: Exozodiacal Disk

Local and exo-Zodiacal light are important astrophysical noise sources that must be considered when estimating integration times. Simulating Zodiacal Point Sources is describe in Notebook 04. The Simulation of an extended source has a similar logic, except here we integrate surface brightness over the coronograph's photometric aperature.


If running via Google Colab, you must first execute the contents of notebook `00_Google_Colab_Setup.ipynb` (only if you have never done so previously). Then execute all cells tagged with &#128992;.  If running via a local installation, you should skip all of the colab-specific (&#128992;) cells.

## 🟠 Setup for Google Colab Use

### 🟠 Run the next cell to mount the Google Drive

You will receive some or all of the following prompts:

* Warning: This notebook was not authored by Google - Click "Run Anyway" 
* Permit this notebook to access your Google Drive files? - Click "Connect to Google Drive"
* A new browser window will prompt you to select an account and authorize access
  * Select the Google account you wish to use and click Continue on each subsequent screen until the dialog vanishes

Upon completion of cell execution, you should see `Mounted at /content/drive`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 🟠 Run the next cell to change to the corgietc directory and install the required software

This process should take less than a minute, but, depending on bandwidth availability, may take as long as a few minutes. You will see a variety of messages about package downloads.  Upon completion of cell execution, you should see `Sucessfully installed` followed by a list of installed packages and their versions.

You may see the prompt "Restart session".  You do not need to do this - click 'Cancel'.

In [ ]:
# This cell should *only* be executed if running the notebook in Google Colab
import os

# Google top level drive dir
drive_dir = "/content/drive/MyDrive/"

# directory path
corgietc_dir = 'corgietc'
corgietc_path = os.path.join(drive_dir, corgietc_dir)
cgi_noise_repo_path = os.path.join(corgietc_path, "cgi_noise")
corgietc_repo_path = os.path.join(corgietc_path, "corgietc")
corgietc_notebooks_path = os.path.join(corgietc_repo_path, "Notebooks")

# Change to the cgi_noise repo path and update the repo
os.chdir(cgi_noise_repo_path)
!git pull

# Install the backend and all requirements - this can also take a little while
!pip install .

# Change to the corgietc repo path and update the repo
os.chdir(corgietc_repo_path)
!git pull

# Install the backend and all requirements - this can also take a little while
!pip install .

# Refresh package list to pick up new installations
import site
site.main()

# Change to the Notebooks directory
os.chdir(corgietc_notebooks_path)

### 🟠 Import jupyter widget for Colab

In [ ]:
# need to import third party jupyter widget
from google.colab import output
output.enable_custom_widget_manager()

## All Cells from this point should be run for both Colab and local installations

In [ ]:
# import all required packages
from corgietc.corgietc import corgietc
import os
import json
import EXOSIMS.Prototypes.TargetList
import EXOSIMS.Prototypes.TimeKeeping
import EXOSIMS.Observatory.ObservatoryL2Halo
import copy
import astropy.units as u
from astropy.time import Time
import numpy as np
import matplotlib.pyplot as plt

## Setup: Building the TargetList

We must start by loading the default `CGI_Noise.json` input specifications, as well as a `TargetList`

In [ ]:
scriptfile = os.path.join(os.environ["CORGIETC_DATA_DIR"], "scripts", "CGI_Noise.json")
with open(scriptfile, "r") as f:
    specs = json.loads(f.read())
specs["modules"]["StarCatalog"] = "HWOMissionStars"
TL = EXOSIMS.Prototypes.TargetList.TargetList(**copy.deepcopy(specs))
ZL = TL.ZodiacalLight
OS = TL.OpticalSystem

In this Notebook, we will work with the first observing mode and 47 UMa (HIP 53721) taget. 

In [ ]:
# Select observing mode and target
mode = OS.observingModes[0]
sInd = np.where(TL.Name == "HIP 53721")[0]

print(f"Observing mode: {mode['Scenario']}")
print(f"Central wavelength: {mode['lam']}")
print(f"Target: {TL.Name[sInd][0]}")

### The Coronagraph Photometric Aperture: `Omega`

For a point source like a planet, the signal is collected within the PSF core. The size of that core — i.e., the photometric aperture — is captured by the coronagraph's `core_area`, which we call `Omega`. It has units of arcsec² and depends on both the observing wavelength and the working angle (separation from the star).

For an **extended source**, `Omega` plays a critical role: it is the solid angle over which the surface brightness is integrated to produce a count rate. The larger `Omega`, the more disk flux is collected.

Let's first define our working angle of interest and inspect the value of `Omega` at that location.

In [ ]:
# Define working angle in lambda/D units, then convert to arcseconds
WA = np.array([7.5]) * (mode["lam"] / OS.pupilDiam).to(
    u.arcsec, equivalencies=u.dimensionless_angles()
)
print(f"Working angle: {WA[0]:.4f}")

# Retrieve the coronagraph core area (photometric aperture) at this WA
syst = mode["syst"]
lam  = mode["lam"]
Omega = syst["core_area"](lam, WA)
print(f"Coronagraph core area Omega = {Omega[0]:.4f}")

We can also see how `Omega` varies as a function of working angle across the full coronagraph field. This illustrates that the effective aperture size changes with separation from the star — something that has no analog in a point-source calculation.

In [ ]:
# Compute Omega across a range of working angles
lam_D = (mode["lam"] / OS.pupilDiam).to(u.arcsec, equivalencies=u.dimensionless_angles())
WA_range = np.linspace(3, 20, 200) * lam_D
Omega_range = syst["core_area"](lam, WA_range)

plt.figure()
plt.plot(WA_range / lam_D, Omega_range)
plt.xlabel(r"Working Angle ($\lambda$/D)")
plt.ylabel(f"Core Area $\\Omega$ ({Omega_range.unit})")
plt.title("Coronagraph Photometric Aperture vs. Working Angle")
plt.axvline(7.5, color='gray', linestyle='--', label='WA = 7.5 λ/D')
plt.legend()
plt.tight_layout();

## Computing the Exozodiacal Surface Brightness

The surface brightness of the exozodiacal dust disk is described by the standard magnitude equation:

$$f_{Z,\mathrm{ez}} = F_0 \times 10^{-m_{\mathrm{EZ}}/2.5} \quad [\mathrm{ph\,s^{-1}\,m^{-2}\,nm^{-1}\,arcsec^{-2}}]$$

where:
- $F_0$ is the zero-magnitude spectral flux density for the observing band (`mode["F0"]`), i.e., the flux corresponding to a 0th magnitude source integrated over the bandpass
- $m_{\mathrm{EZ}} = 22$ mag/arcsec² is the default exozodi surface brightness from [Stark et al. (2014)](http://dx.doi.org/10.1088/0004-637X/795/2/122), also stored in `ZL.magEZ`

This is entirely analogous to the local zodiacal light surface brightness `fZ0` computed from `ZL.magZ` — the only difference is the assumed magnitude value.

In [ ]:
# Default exozodi surface brightness magnitude (from Stark 2014)
magEZ = ZL.magEZ  # 22 mag/arcsec^2
print(f"magEZ = {magEZ} mag/arcsec²")
print(f"For comparison, magZ (local zodi) = {ZL.magZ} mag/arcsec²")

# Compute exozodi surface brightness via the standard magnitude equation
# F0 is the zero-point spectral flux density integrated over the bandpass [ph/s/m^2]
# Dividing by arcsec^2 gives surface brightness [ph/s/m^2/arcsec^2]
fZ_ez = mode["F0"] * 10**(-magEZ / 2.5) / u.arcsec**2
print(f"\nExozodi surface brightness: fZ_ez = {fZ_ez:.4e}")

## Computing the Extended Source Count Rate

Now we have all the pieces to compute the photon count rate from the exozodiacal disk as an extended source. The count rate is:

$$C_{\mathrm{ez}} = f_{Z,\mathrm{ez}} \times \mathtt{losses} \times \Omega \times T_{\mathrm{occ}}$$

where:
- $f_{Z,\mathrm{ez}}$ is the exozodi surface brightness computed above [ph/s/m²/arcsec²]
- `losses` bundles together the telescope collecting area, quantum efficiency, optics throughput, and effective bandwidth — everything between the photon arriving at the primary mirror and being recorded as an electron on the detector
- $\Omega$ is the coronagraph core area (photometric aperture) [arcsec²]
- $T_{\mathrm{occ}}$ is the coronagraph occulter transmission at the working angle of interest

This is exactly the same structure as the zodiacal light count rate `C_z` computed inside `OpticalSystem.Cp_Cb_Csp_helper`:
```python
C_z = mode["F0"] * mode["losses"] * fZ * Omega * occ_trans
```
The difference is that here `fZ_ez` already contains `F0`, so we write `fZ_ez * losses` rather than `F0 * losses * fZ`.

In [ ]:
# Coronagraph occulter transmission at our working angle
occ_trans = syst["occ_trans"](lam, WA)
print(f"Occulter transmission at WA = {WA[0]:.3f}: {occ_trans[0]:.4f}")

# mode["losses"] bundles: pupil area * QE * optics throughput * effective bandwidth
print(f"Mode losses: {mode['losses'].value[0]:.4e} {mode['losses'].unit}")

# Extended source count rate from the exozodiacal disk
C_ez = (fZ_ez * mode["losses"] * Omega * occ_trans).to("1/s")
print(f"\nExtended source count rate: C_ez = {C_ez[0].value:.4e} {C_ez[0].unit}")

## Scaling with Number of Zodis

The value above assumes exactly 1 zodi of exozodiacal dust — i.e., the same dust density as our own solar system. In practice, target systems may have more or less dust. The number of zodis (`nEZ`) scales the surface brightness linearly, so the count rate scales the same way.

Let's see how the count rate changes for different assumed dust levels:

In [ ]:
nEZ_values = np.array([1, 3, 5, 10, 25, 50])

print(f"{'nEZ':>6}  {'C_ez (1/s)':>15}")
print("-" * 25)
for nEZ in nEZ_values:
    C_ez_nEZ = (nEZ * fZ_ez * mode["losses"] * Omega * occ_trans).to("1/s")
    print(f"{nEZ:>6}  {C_ez_nEZ[0].value:>15.4e}")

In [ ]:
# identify 47 UMa c
sInd = np.where(TL.Name == "HIP 53721")[0]
# extract first observing mode from optical system
mode = OS.observingModes[0]
# compute zodi flux
fZ = ZL.fZ(Obs, TL, sInd, ts, mode)
plt.figure()
plt.plot(range(len(ts)), fZ)
plt.xlabel("Days after 1/1/2027")
plt.ylabel(f"fZ for 47 UMa for {mode['Scenario']}");

## Comparing Extended Source vs. Point Source Treatment

It is instructive to compare the extended source count rate we just computed to how the exozodi enters the standard point-source integration time calculation, where it appears as a noise term via `JEZ`.

In that formulation (from `Cp_Cb_Csp_helper`):
```python
C_ez = JEZ * mode["losses"] * Omega * occ_trans
```
where `JEZ` has units of ph/s/m²/arcsec² and is pre-computed by EXOSIMS for each target. The conceptual structure is identical — the difference is simply where the surface brightness value comes from.

In [ ]:
# Retrieve the pre-computed JEZ0 for our target and mode
JEZ0 = TL.JEZ0[mode["hex"]][sInd]
print(f"JEZ0 for {TL.Name[sInd][0]}: {JEZ0[0]:.4e}")
print(f"fZ_ez (our extended source):  {fZ_ez:.4e}")
print()

# Count rate via the point-source noise treatment (as EXOSIMS does it internally)
C_ez_JEZ = (JEZ0 * mode["losses"] * Omega * occ_trans).to("1/s")

# Count rate via our extended source surface brightness formula
C_ez_fZ  = (fZ_ez * mode["losses"] * Omega * occ_trans).to("1/s")

print(f"C_ez via JEZ0 (point-source noise term): {C_ez_JEZ[0]:.4e}")
print(f"C_ez via fZ_ez (extended source):        {C_ez_fZ[0]:.4e}")

## Count Rate as a Function of Working Angle

Because both `Omega` and `occ_trans` vary with working angle, the extended source count rate is not uniform across the coronagraph field. Let's compute and plot this variation.

In [ ]:
# Compute count rate across working angles
WA_range = np.linspace(3, 20, 200) * lam_D

Omega_range    = syst["core_area"](lam, WA_range)
occ_trans_range = syst["occ_trans"](lam, WA_range)

C_ez_range = (fZ_ez * mode["losses"] * Omega_range * occ_trans_range).to("1/s")

plt.figure()
plt.plot(WA_range / lam_D, C_ez_range)
plt.xlabel(r"Working Angle ($\lambda$/D)")
plt.ylabel(r"$C_{\rm ez}$ (1/s)")
plt.title("Extended Source Count Rate vs. Working Angle\n"
          f"(magEZ = {magEZ} mag/arcsec², 1 zodi)")
plt.axvline(7.5, color='gray', linestyle='--', label='WA = 7.5 λ/D')
plt.legend()
plt.tight_layout();

## Summary

The extended source calculation follows the same structure as the point-source background noise calculation in EXOSIMS, but with the surface brightness computed explicitly from the standard magnitude equation:

$$f_{Z,\mathrm{ez}} = F_0 \times 10^{-m_{\mathrm{EZ}}/2.5} \quad [\mathrm{ph\,s^{-1}\,m^{-2}\,nm^{-1}\,arcsec^{-2}}]$$

The count rate from the exozodiacal disk over the coronagraph's photometric aperture is then:

$$C_{\mathrm{ez}} = f_{Z,\mathrm{ez}} \times \mathtt{losses} \times \Omega \times T_{\mathrm{occ}}$$